In [ ]:
pip install pandas scikit-learn xgboost deap nltk joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 7.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re, string, random
import nltk
from nltk.stem import WordNetLemmatizer
from deap import base, creator, tools
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# NLTK resources
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()
random.seed(42)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
# Load dataset
fake = pd.read_csv("data/fake.csv", on_bad_lines='skip', engine='python')
real = pd.read_csv("data/real.csv", on_bad_lines='skip', engine='python')

fake['label'] = 0
real['label'] = 1
df = pd.concat([fake, real]).reset_index(drop=True)

# Cleaning function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www.\S+', '', text)  # Remove URLs
    text = re.sub(r'\S+@\S+', '', text)          # Remove emails
    text = re.sub(r'\d+', '', text)              # Remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()     # Remove extra spaces
    text = " ".join([lemmatizer.lemmatize(word) for word in text.split()])  # Lemmatize
    return text

df['clean_text'] = df['text'].apply(clean_text)
df = df[df['clean_text'].str.len() > 3]
df = df.sample(frac=1, random_state=42).reset_index(drop=True)


FileNotFoundError: [Errno 2] No such file or directory: 'data/fake.csv'

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, stratify=df['label'], random_state=42
)

tfidf = TfidfVectorizer(stop_words='english', max_features=20000)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)


In [ ]:
def build_stacking_model(svm_c=1.0, lr_c=1.0, dt_depth=5):
    """
    Generic reusable stacking pipeline
    """
    base_models = [
        ('svm', LinearSVC(C=svm_c, max_iter=500)),
        ('lr', LogisticRegression(C=lr_c, max_iter=500)),
        ('dt', DecisionTreeClassifier(max_depth=dt_depth))
    ]
    meta_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                               use_label_encoder=False, eval_metric='logloss')

    stack_model = StackingClassifier(estimators=base_models,
                                     final_estimator=meta_model,
                                     passthrough=True)
    return stack_model


In [ ]:
# Create fitness and individual
try:
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)
except:
    pass  # ignore if already created

toolbox = base.Toolbox()

# Genes: [SVM_C, LR_C, DT_depth]
toolbox.register("attr_svm_c", random.uniform, 0.01, 10)
toolbox.register("attr_lr_c", random.uniform, 0.01, 10)
toolbox.register("attr_dt_depth", random.randint, 1, 50)

toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.attr_svm_c, toolbox.attr_lr_c, toolbox.attr_dt_depth), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)


In [ ]:
def evaluate(individual):
    svm_c = max(0.01, individual[0])
    lr_c = max(0.01, individual[1])
    dt_depth = int(max(1, individual[2]))

    model = build_stacking_model(svm_c=svm_c, lr_c=lr_c, dt_depth=dt_depth)
    model.fit(X_train_vec, y_train)
    pred = model.predict(X_test_vec)
    return (accuracy_score(y_test, pred),)


In [ ]:
# GA Operators
toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)

# Run GA
pop = toolbox.population(n=6)  # population size
NGEN = 5  # generations

for gen in range(NGEN):
    offspring = toolbox.select(pop, len(pop))
    offspring = list(map(toolbox.clone, offspring))

    # Crossover
    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < 0.7:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    # Mutation
    for mutant in offspring:
        if random.random() < 0.2:
            toolbox.mutate(mutant)
            del mutant.fitness.values

    # Evaluate fitness
    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = map(toolbox.evaluate, invalid_ind)
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    # Update population
    pop[:] = offspring

    # Print best
    best_ind = tools.selBest(pop, 1)[0]
    print(f"Generation {gen+1} Best Accuracy: {best_ind.fitness.values[0]:.4f}")

best_hyperparams = best_ind


In [ ]:
svm_c, lr_c, dt_depth = best_hyperparams
svm_c = float(svm_c)
lr_c = float(lr_c)
dt_depth = int(dt_depth)

final_model = build_stacking_model(svm_c=svm_c, lr_c=lr_c, dt_depth=dt_depth)
final_model.fit(X_train_vec, y_train)

pred = final_model.predict(X_test_vec)
print("Final Accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred))


In [ ]:
joblib.dump(final_model, "stacked_model.pkl")
joblib.dump(tfidf, "vectorizer.pkl")


In [ ]:
def predict_news(news_text):
    model = joblib.load("stacked_model.pkl")
    vectorizer = joblib.load("vectorizer.pkl")
    clean = clean_text(news_text)
    vec = vectorizer.transform([clean])
    pred = model.predict(vec)[0]
    return "Fake News" if pred == 0 else "Real News"




In [ ]:

print(predict_news(" announces teleportation device."))